# Practice: Transfer Learning with ResNet18

Use this practice after Section 7. It runs independently of the Section 6 practice.

- Prepare image batches using the pretrained weights' transforms.
- Replace the ImageNet head with a ten-class classifier and freeze the backbone.
- Train the head and check which weights and BatchNorm statistics changed.
- Measure accuracy on held-out digits.

Run the notebook from top to bottom in a fresh runtime. It downloads the digit images if needed. The first run also downloads about 45 MB of ResNet18 weights. A GPU is optional.

Use Colab's standard Python runtime. For local Jupyter, install missing packages with `python -m pip install torch torchvision scikit-learn`.

[Open in Colab](https://colab.research.google.com/github/wlos/ai_with_python_course/blob/main/16-Deep-Learning-and-Image-Processing/practice/Practice%20Transfer%20Learning%20with%20ResNet18.ipynb)


## 1. Prepare inputs for ResNet18

Predict how many channels the pretrained model expects. Run the cell to check the image shape. Identify where the code selects the resize, crop and normalisation associated with the pretrained weights.


In [ ]:
from os import makedirs, replace
from os.path import isfile
from shutil import unpack_archive
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve

DIGIT_ARCHIVE = "sample_data/digit_images.zip"
DIGIT_URL = (
    "https://raw.githubusercontent.com/wlos/AI_SS/main/"
    "13-Computer-Vision/sample_data/digit_images.zip"
)

if not isfile(DIGIT_ARCHIVE):
    makedirs("sample_data", exist_ok=True)
    urlretrieve(DIGIT_URL, DIGIT_ARCHIVE + ".part")
    replace(DIGIT_ARCHIVE + ".part", DIGIT_ARCHIVE)  # Keep only a completed download.

# Keep the extracted images available while ImageFolder reads them on demand.
digit_files = TemporaryDirectory()
unpack_archive(DIGIT_ARCHIVE, digit_files.name)
digit_root = f"{digit_files.name}/sample_data/digit_images"


from torchvision import datasets
from torchvision.models import ResNet18_Weights

resnet_weights = ResNet18_Weights.DEFAULT
digit_dataset = datasets.ImageFolder(digit_root, transform=resnet_weights.transforms())
resnet_image, label = digit_dataset[0]
print(f"Input shape: {tuple(resnet_image.shape)} | Label: {label}")
assert len(digit_dataset) == 240 and len(digit_dataset.classes) == 10
assert tuple(resnet_image.shape) == (3, 224, 224)


## 2. Split and batch the images

Use the same stratified 80/20 split and seed as in Section 6. Record the batch shape and explain why the same image files produce different input tensors for ResNet18 and for the simple CNN.


In [ ]:
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

DIGIT_SEED = 42
DIGIT_TEST_FRACTION = 0.2
DIGIT_BATCH_SIZE = 16

# Split image indices so the dataset still opens and transforms each file on demand.
digit_train_indices, digit_test_indices = train_test_split(
    list(range(len(digit_dataset))),
    test_size=DIGIT_TEST_FRACTION,
    random_state=DIGIT_SEED,
    stratify=digit_dataset.targets,
)
digit_train = Subset(digit_dataset, digit_train_indices)
digit_test = Subset(digit_dataset, digit_test_indices)
digit_train_loader = DataLoader(
    digit_train, batch_size=DIGIT_BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(DIGIT_SEED),
)
digit_test_loader = DataLoader(digit_test, batch_size=DIGIT_BATCH_SIZE, shuffle=False)

batch_images, batch_labels = next(iter(digit_test_loader))
print(f"Training images: {len(digit_train)} | Test images: {len(digit_test)}")
print(f"Image batch: {tuple(batch_images.shape)} | Label batch: {tuple(batch_labels.shape)}")
print(f"Batch labels: {batch_labels.tolist()}")


## 3. Set up the trainable head

Identify which line freezes the existing parameters and which line selects the parameters for Adam. Use the trainable parameter names to check that the classifier has a weight matrix and a bias vector. Explain how `eval()` protects BatchNorm running statistics while the optimiser updates the new head.


In [ ]:
from torchvision.models import resnet18

from torch import nn

DIGIT_LEARNING_RATE = 0.001
digit_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
digit_class_count = len(digit_dataset.classes)
torch.manual_seed(DIGIT_SEED)
digit_resnet = resnet18(weights=resnet_weights)
digit_resnet.requires_grad_(False)  # Freeze all original pretrained parameters.
digit_resnet.fc = nn.Linear(digit_resnet.fc.in_features, digit_class_count)
digit_resnet = digit_resnet.to(digit_device)
digit_resnet.eval()  # Preserve the backbone's BatchNorm running statistics.
digit_resnet.fc.train()  # Set training mode only for the replacement head.
resnet_optimiser = torch.optim.Adam(digit_resnet.fc.parameters(), lr=DIGIT_LEARNING_RATE)
print(f"Trainable head parameters: {sum(p.numel() for p in digit_resnet.fc.parameters()):,}")

trainable_names = [name for name, parameter in digit_resnet.named_parameters()
                   if parameter.requires_grad]
print("Trainable parameters by name:", trainable_names)
assert trainable_names == ["fc.weight", "fc.bias"]

# Save copies so changes can be checked after training.
head_before = digit_resnet.fc.weight.detach().clone()
backbone_before = digit_resnet.conv1.weight.detach().clone()
running_mean_before = digit_resnet.bn1.running_mean.detach().clone()


## 4. Train and verify which values changed

Before running the next cells, predict whether each of the three saved tensors will match after training. Record the head's loss across epochs.


In [ ]:
DIGIT_EPOCHS = 3
digit_loss = nn.CrossEntropyLoss()


def train_digit_epoch(model, loader, optimiser):
    '''Update trainable parameters and return mean loss per image.'''
    total_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(digit_device), labels.to(digit_device)
        optimiser.zero_grad()  # Clear gradients left by the previous batch.
        loss = digit_loss(model(images), labels)
        loss.backward()  # Calculate gradients from the loss.
        optimiser.step()  # Apply an update to the optimiser's parameters.
        total_loss += loss.item() * len(labels)  # Weight each batch by its size.
    return total_loss / len(loader.dataset)


# Restore the intended modes if you rerun this cell after evaluating.
digit_resnet.eval()
digit_resnet.fc.train()
for epoch in range(DIGIT_EPOCHS):
    mean_loss = train_digit_epoch(digit_resnet, digit_train_loader, resnet_optimiser)
    print(f"Head epoch {epoch + 1}: loss={mean_loss:.3f}")


In [ ]:
head_changed = not torch.equal(digit_resnet.fc.weight, head_before)
backbone_changed = not torch.equal(digit_resnet.conv1.weight, backbone_before)
running_mean_changed = not torch.equal(digit_resnet.bn1.running_mean, running_mean_before)
print(f"Head weight changed: {head_changed}")
print(f"First convolution weight changed: {backbone_changed}")
print(f"First BatchNorm running mean changed: {running_mean_changed}")
assert head_changed and not backbone_changed and not running_mean_changed


## 5. Evaluate on held-out digits

Record the accuracy on the 48 test images. Compare it with the simple CNN result if you completed Section 6. Name one reason ImageNet features might help and one reason a small digit test set can make the comparison uncertain.


In [ ]:
def digit_accuracy(model, loader):
    '''Measure the fraction of test images classified correctly.'''
    model.eval()  # Use each layer's inference behaviour.
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            predicted = model(images.to(digit_device)).argmax(dim=1)
            correct += (predicted.cpu() == labels).sum().item()
    return correct / len(loader.dataset)


resnet_digit_accuracy = digit_accuracy(digit_resnet, digit_test_loader)
print(f"Frozen ResNet18 test accuracy: {resnet_digit_accuracy:.1%}")
